In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 🏅 Medallion Architecture Mock — Bronze / Silver / Gold
# MAGIC
# MAGIC Gera **5 milhões de linhas** de dados de vendas (e-commerce) e processa
# MAGIC em três camadas usando **PySpark + Delta Lake**.
# MAGIC
# MAGIC - **Bronze** → dados crus, "sujos" (nulos, duplicados, tipos errados, datas em texto)
# MAGIC - **Silver** → dados limpos, tipados, deduplicados e validados
# MAGIC - **Gold** → tabelas agregadas prontas para BI / análise
# MAGIC
# MAGIC Feito para rodar no **Databricks Free Edition** (serverless + Unity Catalog).

# COMMAND ----------

# MAGIC %md
# MAGIC ## 0. Configuração — catálogo e schema

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql import types as T

# No Free Edition o catálogo padrão é "workspace".
CATALOG = "workspace"
SCHEMA  = "medallion_demo"
N_ROWS  = 5_000_000   # 5 milhões

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Usando {CATALOG}.{SCHEMA} | gerando {N_ROWS:,} linhas")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. 🥇 GOLD — agregações de negócio
# MAGIC
# MAGIC Três tabelas analíticas a partir da Silver.

# COMMAND ----------

silver = spark.table("silver_sales")

# 3.1 Receita por categoria e país
gold_cat = (
    silver.groupBy("category", "country")
    .agg(
        F.count("*").alias("num_transactions"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_ticket"),
        F.sum("quantity").alias("units_sold"),
    )
    .orderBy(F.col("revenue").desc())
)
gold_cat.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable("gold_revenue_by_category")

# 3.2 Receita mensal
gold_month = (
    silver
    .withColumn("year_month", F.date_format("event_date", "yyyy-MM"))
    .groupBy("year_month")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.count("*").alias("num_transactions"),
        F.countDistinct("customer_id").alias("active_customers"),
    )
    .orderBy("year_month")
)
gold_month.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable("gold_monthly_revenue")

# 3.3 Top clientes (RFM simplificado)
gold_cust = (
    silver.groupBy("customer_id")
    .agg(
        F.count("*").alias("frequency"),
        F.round(F.sum("total_amount"), 2).alias("monetary"),
        F.max("event_date").alias("last_purchase"),
    )
    .orderBy(F.col("monetary").desc())
)
gold_cust.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable("gold_customer_value")

#print("Gold gravada ✅")
#print("\n— Receita por categoria/país —")
#display(spark.table("gold_revenue_by_category").limit(20))
#print("\n— Receita mensal —")
#display(spark.table("gold_monthly_revenue"))
#print("\n— Top clientes —")
#display(spark.table("gold_customer_value").limit(20))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. ✅ Resumo / validação das camadas

# COMMAND ----------

#for t in ["bronze_sales", "silver_sales",
#          "gold_revenue_by_category", "gold_monthly_revenue", "gold_customer_value"]:
#    print(f"{t:32s} -> {spark.table(t).count():>10,} linhas")

# COMMAND ----------

# MAGIC %md
# MAGIC ### (opcional) Limpeza — apaga tudo
# MAGIC ```python
# MAGIC spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")
# MAGIC ```